# Homework 7 - BERT Question Answering

**任務**：中文抽取式問答（Extractive QA）
- 輸入：段落 + 問題
- 輸出：答案在段落中的起始/結束位置

**改進重點（相對於 Simple baseline）**：
- **Medium**：`doc_stride` 縮小（重疊 window）+ Linear LR decay with warmup
- **Strong**：換用 `hfl/chinese-roberta-wwm-ext` + Preprocessing 修正 None token
- **Boss**：Postprocessing 修正（限制預測在段落範圍、確保 end >= start）

## 1. 下載資料集

In [ ]:
# 下載資料集
!gdown --id '1znKmX08v9Fygp-dgwo7BKiLIf2qL1FH1' --output hw7_data.zip

# 若上面失敗，改用備用連結
# !gdown --id '1pOu3FdPdvzielUZyggeD7KDnVy9iW1uC' --output hw7_data.zip

!unzip -o hw7_data.zip

# 確認 GPU 型號（V100 > T4 > P4 > K80）
!nvidia-smi

## 2. 安裝套件

In [ ]:
# transformers：HuggingFace 預訓練模型工具包
# accelerate：支援 fp16 混合精度訓練（加速約 2-3x）
!pip install transformers==4.5.0 accelerate==0.2.0

## 3. Import 套件

In [ ]:
import json
import numpy as np
import random
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AdamW,
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用裝置：{device}")

# 固定亂數種子，確保結果可重現
def same_seeds(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

same_seeds(0)

## 4. FP16 混合精度設定

In [ ]:
# fp16（半精度）能大幅縮短訓練時間，建議開啟
# T4 GPU 上 fp16 訓練速度約為 fp32 的 2-3 倍
fp16_training = True

if fp16_training:
    from accelerate import Accelerator
    accelerator = Accelerator(fp16=True)
    device = accelerator.device
    print(f"FP16 啟用，裝置：{device}")

## 5. 載入預訓練模型

**Strong baseline 改動**：換用 `hfl/chinese-roberta-wwm-ext`
- 相比 `bert-base-chinese`，這個模型使用 Whole Word Masking（全詞遮蔽）預訓練
- 在中文 NLP 任務上普遍有顯著提升

In [ ]:
# Strong: 使用更強的中文預訓練模型
# bert-base-chinese 公開分數約 0.446（Simple），換模型後能達到 Strong 以上
model_name = "hfl/chinese-roberta-wwm-ext"

model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"模型載入完成：{model_name}")
# 警告訊息可以忽略（QA head 是隨機初始化的，因為是新任務的輸出層）

## 6. 讀取資料

資料格式：
- `questions`：問題列表，每個問題包含 id、paragraph_id、question_text、answer_text、answer_start、answer_end
- `paragraphs`：段落列表（測試集的 answer 欄位為 null）

In [ ]:
def read_data(file):
    with open(file, 'r', encoding="utf-8") as reader:
        data = json.load(reader)
    return data["questions"], data["paragraphs"]

train_questions, train_paragraphs = read_data("hw7_train.json")
dev_questions, dev_paragraphs = read_data("hw7_dev.json")
test_questions, test_paragraphs = read_data("hw7_test.json")

print(f"訓練集問題數：{len(train_questions)}")
print(f"驗證集問題數：{len(dev_questions)}")
print(f"測試集問題數：{len(test_questions)}")

## 7. Tokenize 資料

先分別 tokenize 問題和段落（不加 special tokens），
之後在 Dataset 的 `__getitem__` 再組合並加上 [CLS]、[SEP]。

In [ ]:
# add_special_tokens=False：不自動加 [CLS]/[SEP]，之後手動控制
train_questions_tokenized = tokenizer(
    [q["question_text"] for q in train_questions], add_special_tokens=False
)
dev_questions_tokenized = tokenizer(
    [q["question_text"] for q in dev_questions], add_special_tokens=False
)
test_questions_tokenized = tokenizer(
    [q["question_text"] for q in test_questions], add_special_tokens=False
)

train_paragraphs_tokenized = tokenizer(train_paragraphs, add_special_tokens=False)
dev_paragraphs_tokenized = tokenizer(dev_paragraphs, add_special_tokens=False)
test_paragraphs_tokenized = tokenizer(test_paragraphs, add_special_tokens=False)

print("Tokenize 完成")
# 警告訊息可以忽略（序列超過 512 會警告，但我們在 Dataset 中會切 window）

## 8. Dataset 與 DataLoader

**關鍵概念：Sliding Window**
- BERT 最大輸入長度 512（Self-Attention 是 O(n²)，太長會爆記憶體）
- 很多段落超過 512，需要用 sliding window 把段落切成多個小片段
- **訓練**：只取包含答案的那個 window
- **測試**：切成多個 window 各自預測，取信心分數最高的答案

**Medium 改動**：`doc_stride` 150 → 75，讓相鄰 window 有重疊
- 若答案在兩個 window 的邊界，重疊可確保至少有一個 window 完整包含答案

**Strong 改動**：Preprocessing 修正 `char_to_token` 可能回傳 `None` 的問題

In [ ]:
class QA_Dataset(Dataset):
    def __init__(self, split, questions, tokenized_questions, tokenized_paragraphs):
        self.split = split
        self.questions = questions
        self.tokenized_questions = tokenized_questions
        self.tokenized_paragraphs = tokenized_paragraphs
        self.max_question_len = 40
        self.max_paragraph_len = 150
        # Medium: doc_stride 從 150 縮小到 75，讓相鄰 window 重疊 75 個 token
        # 原本 doc_stride=max_paragraph_len 代表完全不重疊
        # 縮小後答案在邊界的情況能被正確預測
        self.doc_stride = 75
        # 完整輸入序列長度 = [CLS] + question + [SEP] + paragraph + [SEP]
        self.max_seq_len = 1 + self.max_question_len + 1 + self.max_paragraph_len + 1

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        question = self.questions[idx]
        tokenized_question = self.tokenized_questions[idx]
        tokenized_paragraph = self.tokenized_paragraphs[question["paragraph_id"]]

        if self.split == "train":
            # 將答案的字元位置轉換成 token 位置
            answer_start_token = tokenized_paragraph.char_to_token(question["answer_start"])
            answer_end_token = tokenized_paragraph.char_to_token(question["answer_end"])

            # Strong Preprocessing: char_to_token 偶爾回傳 None
            # 發生原因：答案結尾字元可能對應到空白或被前一個 token 吸收
            # 解法：往前找最近的有效 token position
            if answer_end_token is None:
                for offset in range(1, 10):
                    answer_end_token = tokenized_paragraph.char_to_token(
                        question["answer_end"] - offset
                    )
                    if answer_end_token is not None:
                        break

            # 若還是 None，跳過此筆資料（用 0,0 佔位）
            if answer_start_token is None or answer_end_token is None:
                answer_start_token = 0
                answer_end_token = 0

            # 以答案中心為準，取一個最大的 window
            # 假設：答案附近的資訊足夠回答問題
            mid = (answer_start_token + answer_end_token) // 2
            paragraph_start = max(
                0,
                min(mid - self.max_paragraph_len // 2,
                    len(tokenized_paragraph) - self.max_paragraph_len)
            )
            paragraph_end = paragraph_start + self.max_paragraph_len

            # 加上 special tokens（101 = [CLS]，102 = [SEP]）
            input_ids_question = [101] + tokenized_question.ids[:self.max_question_len] + [102]
            input_ids_paragraph = tokenized_paragraph.ids[paragraph_start:paragraph_end] + [102]

            # 將答案位置從「整段段落的 token index」轉換成「window 內的 index」
            answer_start_token += len(input_ids_question) - paragraph_start
            answer_end_token += len(input_ids_question) - paragraph_start

            input_ids, token_type_ids, attention_mask = self.padding(
                input_ids_question, input_ids_paragraph
            )
            return (
                torch.tensor(input_ids),
                torch.tensor(token_type_ids),
                torch.tensor(attention_mask),
                answer_start_token,
                answer_end_token,
            )

        else:  # dev / test
            input_ids_list, token_type_ids_list, attention_mask_list = [], [], []

            # 測試時不知道答案在哪，將整段段落切成多個 window
            # doc_stride=75 表示每個 window 起始點間距 75 token
            for i in range(0, len(tokenized_paragraph), self.doc_stride):
                input_ids_question = [101] + tokenized_question.ids[:self.max_question_len] + [102]
                input_ids_paragraph = tokenized_paragraph.ids[i:i + self.max_paragraph_len] + [102]
                input_ids, token_type_ids, attention_mask = self.padding(
                    input_ids_question, input_ids_paragraph
                )
                input_ids_list.append(input_ids)
                token_type_ids_list.append(token_type_ids)
                attention_mask_list.append(attention_mask)

            return (
                torch.tensor(input_ids_list),
                torch.tensor(token_type_ids_list),
                torch.tensor(attention_mask_list),
            )

    def padding(self, input_ids_question, input_ids_paragraph):
        # 不足 max_seq_len 的部分補 0（padding）
        padding_len = self.max_seq_len - len(input_ids_question) - len(input_ids_paragraph)
        # input_ids：token id 序列
        input_ids = input_ids_question + input_ids_paragraph + [0] * padding_len
        # token_type_ids：0 = 問題部分，1 = 段落部分
        token_type_ids = (
            [0] * len(input_ids_question)
            + [1] * len(input_ids_paragraph)
            + [0] * padding_len
        )
        # attention_mask：1 = 真實 token，0 = padding（不需要 attention）
        attention_mask = (
            [1] * (len(input_ids_question) + len(input_ids_paragraph))
            + [0] * padding_len
        )
        return input_ids, token_type_ids, attention_mask


train_set = QA_Dataset("train", train_questions, train_questions_tokenized, train_paragraphs_tokenized)
dev_set = QA_Dataset("dev", dev_questions, dev_questions_tokenized, dev_paragraphs_tokenized)
test_set = QA_Dataset("test", test_questions, test_questions_tokenized, test_paragraphs_tokenized)

train_batch_size = 16

train_loader = DataLoader(train_set, batch_size=train_batch_size, shuffle=True, pin_memory=True)
# dev/test 的 batch_size 固定為 1：一個 batch = 同一問題的所有 window
dev_loader = DataLoader(dev_set, batch_size=1, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False, pin_memory=True)

print(f"訓練集 batches：{len(train_loader)}")

## 9. Evaluate 函式

**Boss Postprocessing 修正**：

原版 bug：
1. `start_index` 可能 > `end_index`（在 logits 各自取 argmax，沒有考慮相對順序）
2. 預測可能落在問題部分，而非段落部分

修正方式：
1. 找到 [SEP] 位置，只在段落範圍內預測
2. 確定 start 位置後，end 只考慮 >= start 的位置

In [ ]:
def evaluate(data, output):
    answer = ''
    max_prob = float('-inf')
    num_of_windows = data[0].shape[1]

    for k in range(num_of_windows):
        input_ids = data[0][0][k]  # shape: (max_seq_len,)

        # Boss: 找到段落的起始/結束位置
        # 輸入格式：[CLS] question [SEP] paragraph [SEP]
        # 第一個 [SEP]（token id=102）之後才是段落
        sep_positions = (input_ids == 102).nonzero(as_tuple=False).squeeze(dim=1)
        if sep_positions.numel() >= 2:
            paragraph_start_pos = sep_positions[0].item() + 1  # 第一個 [SEP] 後
            paragraph_end_pos = sep_positions[1].item()         # 第二個 [SEP] 前
        else:
            # 防禦性處理：若找不到兩個 [SEP]，使用全序列
            paragraph_start_pos = 1
            paragraph_end_pos = len(input_ids) - 1

        start_logits = output.start_logits[k].clone()
        end_logits = output.end_logits[k].clone()

        # Boss: 將段落以外的位置設為 -inf，避免預測到問題或 padding
        start_logits[:paragraph_start_pos] = float('-inf')
        end_logits[:paragraph_start_pos] = float('-inf')
        start_logits[paragraph_end_pos:] = float('-inf')
        end_logits[paragraph_end_pos:] = float('-inf')

        start_prob, start_index = torch.max(start_logits, dim=0)

        # Boss: 確保 end >= start，避免倒序答案
        end_logits[:start_index] = float('-inf')
        end_prob, end_index = torch.max(end_logits, dim=0)

        prob = start_prob + end_prob

        if prob > max_prob:
            max_prob = prob
            # 將 token ids 轉回文字（中文 BERT tokenizer 會加空格，需去除）
            answer = tokenizer.decode(data[0][0][k][start_index:end_index + 1])

    return answer.replace(' ', '')

## 10. 訓練

**Medium 改動**：Linear LR decay with warmup
- Warmup：前 10% 的 steps 讓 LR 從 0 線性上升到設定值
- Decay：之後讓 LR 從設定值線性下降到 0
- 好處：避免訓練初期梯度爆炸，訓練後期精細調整參數

In [ ]:
num_epoch = 3          # 訓練 3 個 epoch（Simple 只跑 1 epoch）
validation = True      # 每個 epoch 結束後在 dev set 上評估
logging_step = 100     # 每 100 步印一次 loss/acc
learning_rate = 1e-4   # 初始 Learning Rate

optimizer = AdamW(model.parameters(), lr=learning_rate)

# Medium: Linear LR decay with warmup
# 總步數 = epoch 數 × 每 epoch 的 batch 數
total_steps = num_epoch * len(train_loader)
# Warmup 佔前 10% 的步數
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"總訓練步數：{total_steps}，Warmup 步數：{warmup_steps}")

if fp16_training:
    # accelerator.prepare 會自動處理 fp16 的梯度縮放
    model, optimizer, train_loader = accelerator.prepare(model, optimizer, train_loader)

model.train()
print("開始訓練...")

for epoch in range(num_epoch):
    step = 1
    train_loss = train_acc = 0

    for data in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
        # 將所有資料移到 GPU
        data = [i.to(device) for i in data]

        # BERT QA 模型輸入
        # 提供 start_positions/end_positions 時，模型會自動計算 Cross Entropy Loss
        output = model(
            input_ids=data[0],
            token_type_ids=data[1],
            attention_mask=data[2],
            start_positions=data[3],
            end_positions=data[4],
        )

        # 計算訓練準確率（start 和 end 都正確才算對）
        start_index = torch.argmax(output.start_logits, dim=1)
        end_index = torch.argmax(output.end_logits, dim=1)
        train_acc += ((start_index == data[3]) & (end_index == data[4])).float().mean()
        train_loss += output.loss

        # 反向傳播
        if fp16_training:
            accelerator.backward(output.loss)
        else:
            output.loss.backward()

        optimizer.step()
        scheduler.step()  # Medium: 每步更新 LR
        optimizer.zero_grad()
        step += 1

        if step % logging_step == 0:
            print(
                f"Epoch {epoch + 1} | Step {step} "
                f"| loss = {train_loss.item() / logging_step:.3f} "
                f"| acc = {train_acc / logging_step:.3f} "
                f"| lr = {scheduler.get_last_lr()[0]:.2e}"
            )
            train_loss = train_acc = 0

    if validation:
        print("評估 Dev Set...")
        model.eval()
        with torch.no_grad():
            dev_acc = 0
            for i, data in enumerate(tqdm(dev_loader, desc="Dev")):
                output = model(
                    input_ids=data[0].squeeze(dim=0).to(device),
                    token_type_ids=data[1].squeeze(dim=0).to(device),
                    attention_mask=data[2].squeeze(dim=0).to(device),
                )
                # Exact Match：預測答案必須完全一致才算正確
                dev_acc += evaluate(data, output) == dev_questions[i]["answer_text"]
            print(f"Validation | Epoch {epoch + 1} | acc = {dev_acc / len(dev_loader):.3f}")
        model.train()

# 儲存模型
print("儲存模型...")
model_save_dir = "saved_model"
model.save_pretrained(model_save_dir)
print(f"模型已儲存至 {model_save_dir}/")

## 11. 測試並輸出結果

In [ ]:
print("評估 Test Set...")
result = []

model.eval()
with torch.no_grad():
    for data in tqdm(test_loader, desc="Test"):
        output = model(
            input_ids=data[0].squeeze(dim=0).to(device),
            token_type_ids=data[1].squeeze(dim=0).to(device),
            attention_mask=data[2].squeeze(dim=0).to(device),
        )
        result.append(evaluate(data, output))

# 輸出 CSV（Kaggle 提交格式）
result_file = "result.csv"
with open(result_file, 'w') as f:
    f.write("ID,Answer\n")
    for i, test_question in enumerate(test_questions):
        # 答案中的逗號需去掉（因為 CSV 用逗號分隔）
        f.write(f"{test_question['id']},{result[i].replace(',', '')}\n")

print(f"完成！結果已寫入 {result_file}")